<img src="../Snowflake_Logo.svg" width="200">

# Itron Intelligence Agent - ML Model Training

This notebook trains and validates the machine learning models used by the Itron Intelligence Agent:

1. **Energy Demand Forecasting** - Time-series prediction of electricity demand
2. **Water Leak Detection** - Classification of anomalous water consumption patterns
3. **Equipment Failure Prediction** - Survival analysis for asset health
4. **Carbon Emissions Forecasting** - Regression for emissions projections
5. **Anomaly Detection** - Isolation Forest for multi-type meter anomalies

**Note:** The agent uses SQL-based UDFs (see `sql/models/09_ml_model_functions.sql`) that implement statistical models directly in Snowflake. This notebook provides the training framework for more sophisticated models that can be registered in the Snowflake Model Registry.

In [ ]:
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, avg, sum as sum_, count, max as max_, min as min_, stddev
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [ ]:
# Establish Snowpark Session (update with your credentials)
session = Session.builder.configs({
    "account": "<your_account>",
    "user": "<your_user>",
    "password": "<your_password>",
    "warehouse": "ITRON_WH",
    "database": "ITRON_DB",
    "schema": "ANALYTICS"
}).create()

print(f"Connected to: {session.get_current_account()}")

## 1. Energy Demand Forecasting
Uses historical meter readings to predict future energy demand with seasonal adjustments.

In [ ]:
# Load electric meter reading data aggregated by day
demand_df = session.sql("""
    SELECT DATE(r.READING_TIMESTAMP) AS READING_DATE, m.REGION,
           SUM(r.READING_VALUE) AS DAILY_KWH, MAX(r.DEMAND_KW) AS PEAK_DEMAND_KW,
           AVG(r.TEMPERATURE_F) AS AVG_TEMP_F, COUNT(DISTINCT m.METER_ID) AS ACTIVE_METERS
    FROM ITRON_DB.RAW.METER_READINGS r
    JOIN ITRON_DB.RAW.METERS m ON r.METER_ID = m.METER_ID
    WHERE m.METER_TYPE = 'ELECTRIC' AND r.QUALITY_FLAG = 'VALID'
    GROUP BY DATE(r.READING_TIMESTAMP), m.REGION ORDER BY READING_DATE
""").to_pandas()

print(f"Demand dataset shape: {demand_df.shape}")
demand_df.head()

In [ ]:
# Feature engineering for demand forecasting
demand_df['DAY_OF_WEEK'] = pd.to_datetime(demand_df['READING_DATE']).dt.dayofweek
demand_df['DAY_OF_YEAR'] = pd.to_datetime(demand_df['READING_DATE']).dt.dayofyear
demand_df['MONTH'] = pd.to_datetime(demand_df['READING_DATE']).dt.month
demand_df['IS_WEEKEND'] = (demand_df['DAY_OF_WEEK'] >= 5).astype(int)
demand_df['SEASONAL_FACTOR'] = np.sin((demand_df['DAY_OF_YEAR'] - 172) * np.pi / 182.5)

print("Features engineered successfully")
print(demand_df[['READING_DATE','REGION','DAILY_KWH','SEASONAL_FACTOR']].describe())

## 2. Water Leak Detection
Classifies water meters as normal or potentially leaking based on consumption pattern deviations.

In [ ]:
# Load water meter statistics for leak analysis
water_df = session.sql("""
    SELECT m.METER_ID, m.REGION,
           AVG(r.READING_VALUE) AS AVG_READING, STDDEV(r.READING_VALUE) AS STD_READING,
           AVG(r.FLOW_RATE) AS AVG_FLOW, AVG(r.PRESSURE_PSI) AS AVG_PRESSURE,
           COUNT(*) AS READING_COUNT,
           AVG(CASE WHEN EXTRACT(HOUR FROM r.READING_TIMESTAMP) BETWEEN 0 AND 5
               THEN r.READING_VALUE ELSE NULL END) AS NIGHT_FLOW_AVG
    FROM ITRON_DB.RAW.METERS m
    JOIN ITRON_DB.RAW.METER_READINGS r ON m.METER_ID = r.METER_ID
    WHERE m.METER_TYPE = 'WATER'
    GROUP BY m.METER_ID, m.REGION HAVING COUNT(*) > 100
""").to_pandas()

water_df['CV'] = water_df['STD_READING'] / water_df['AVG_READING']
water_df['NIGHT_RATIO'] = water_df['NIGHT_FLOW_AVG'] / water_df['AVG_READING']
water_df['LEAK_SCORE'] = 0.4 * np.clip(water_df['NIGHT_RATIO']/0.5, 0, 1) + 0.3 * np.clip(water_df['CV']/2.0, 0, 1) + 0.3 * 0.1

print(f"Water meters analyzed: {len(water_df)}")
print(f"Potential leaks (score > 0.6): {(water_df['LEAK_SCORE'] > 0.6).sum()}")

## 3. Equipment Failure Prediction
Uses a survival model approach based on asset age, condition, and maintenance history.

In [ ]:
# Load asset health data
asset_df = session.sql("""
    SELECT a.ASSET_ID, a.ASSET_TYPE, a.REGION,
           DATEDIFF('year', a.INSTALL_DATE, CURRENT_DATE()) AS AGE_YEARS,
           a.EXPECTED_LIFETIME_YEARS, a.CONDITION_SCORE,
           DATEDIFF('day', a.LAST_INSPECTION_DATE, CURRENT_DATE()) AS DAYS_SINCE_INSPECTION,
           COUNT(DISTINCT wo.WORK_ORDER_ID) AS TOTAL_WORK_ORDERS,
           SUM(CASE WHEN wo.ORDER_TYPE IN ('CORRECTIVE','EMERGENCY') THEN 1 ELSE 0 END) AS FAILURE_ORDERS
    FROM ITRON_DB.RAW.GRID_ASSETS a
    LEFT JOIN ITRON_DB.RAW.WORK_ORDERS wo ON a.ASSET_ID = wo.ASSET_ID
    GROUP BY a.ASSET_ID, a.ASSET_TYPE, a.REGION, a.INSTALL_DATE, a.EXPECTED_LIFETIME_YEARS, a.CONDITION_SCORE, a.LAST_INSPECTION_DATE
""").to_pandas()

asset_df['LIFE_RATIO'] = asset_df['AGE_YEARS'] / asset_df['EXPECTED_LIFETIME_YEARS']
asset_df['FAILURE_PROB'] = (0.3*(1-asset_df['CONDITION_SCORE']/100) + 0.3*np.clip(asset_df['LIFE_RATIO'],0,1.5) + 0.2*np.clip(asset_df['FAILURE_ORDERS']/5,0,1) + 0.2*np.clip(asset_df['DAYS_SINCE_INSPECTION']/365,0,1))

print(f"Assets analyzed: {len(asset_df)}")
print(f"High-risk (>0.7): {(asset_df['FAILURE_PROB'] > 0.7).sum()}")
print(f"Medium-risk (0.5-0.7): {((asset_df['FAILURE_PROB'] > 0.5) & (asset_df['FAILURE_PROB'] <= 0.7)).sum()}")

## 4. Carbon Emissions Forecasting
Projects future emissions based on historical trends and SBTi reduction targets.

In [ ]:
# Load emissions trend data
emissions_df = session.sql("""
    SELECT SCOPE, REPORTING_YEAR, SUM(CO2E_METRIC_TONS) AS ANNUAL_CO2E
    FROM ITRON_DB.ANALYTICS.CARBON_EMISSIONS
    GROUP BY SCOPE, REPORTING_YEAR ORDER BY SCOPE, REPORTING_YEAR
""").to_pandas()

for scope in emissions_df['SCOPE'].unique():
    scope_data = emissions_df[emissions_df['SCOPE'] == scope]
    latest = scope_data.iloc[-1]['ANNUAL_CO2E']
    earliest = scope_data.iloc[0]['ANNUAL_CO2E']
    reduction = (1 - latest / earliest) * 100 if earliest > 0 else 0
    print(f"{scope}: Total reduction = {reduction:.1f}%")

## 5. Anomaly Detection
Multi-type anomaly detection using statistical methods.

In [ ]:
# Statistical anomaly detection
anomaly_df = session.sql("""
    SELECT m.METER_ID, m.METER_TYPE, m.REGION,
           AVG(r.READING_VALUE) AS MEAN_VALUE, STDDEV(r.READING_VALUE) AS STD_VALUE,
           COUNT(*) AS READING_COUNT,
           SUM(CASE WHEN r.TAMPER_FLAG = TRUE THEN 1 ELSE 0 END) AS TAMPER_COUNT
    FROM ITRON_DB.RAW.METERS m
    JOIN ITRON_DB.RAW.METER_READINGS r ON m.METER_ID = r.METER_ID
    WHERE r.READING_TIMESTAMP >= DATEADD('day', -30, CURRENT_TIMESTAMP())
    GROUP BY m.METER_ID, m.METER_TYPE, m.REGION HAVING COUNT(*) > 20
""").to_pandas()

anomaly_df['Z_SCORE'] = np.abs(anomaly_df['MEAN_VALUE'] - anomaly_df.groupby('METER_TYPE')['MEAN_VALUE'].transform('mean')) / anomaly_df.groupby('METER_TYPE')['MEAN_VALUE'].transform('std')
anomaly_df['IS_ANOMALY'] = (anomaly_df['Z_SCORE'] > 2.5) | (anomaly_df['TAMPER_COUNT'] > 0)

print(f"Total meters: {len(anomaly_df)}, Anomalies: {anomaly_df['IS_ANOMALY'].sum()}")

In [ ]:
session.close()
print("Session closed. Models implemented as SQL UDFs for agent use.")